# jjh

---

**Project**
- Unknown-Column-Encoding

**Module**
- notebooks

**Author**
- Hyeok

**Created**
- 2026-03-07

**Purpose**
- TODO: Encoding Unknown Columns

---


### 1. 분석 환경 설정 및 데이터 로드
전 단계(EDA)에서 정제된 데이터를 불러와 본격적인 머신러닝 모델링을 위한 파이프라인을 구축합니다.

비즈니스 목적: 분석에 적합한 최종 데이터셋의 무결성을 확인합니다.

기대 효과: 데이터 로드 단계에서 발생할 수 있는 경로 오류나 데이터 손실을 방지합니다.

In [ ]:
import pandas as pd
import numpy as np

# [Why] 분석용 최종 데이터셋 로드
# 배포 시 주의: 파일 경로('./data/')가 실제 운영 서버 환경과 일치하는지 확인이 필요합니다.
try:
    df = pd.read_csv('./data/df_after_eda.csv')
    print(f"✅ 데이터 로드 완료 (Shape: {df.shape})")
except FileNotFoundError:
    print("❌ 에러: 파일을 찾을 수 없습니다. 경로를 확인해주세요.")

# 데이터 구조 재확인
display(df.head())

✅ 원핫 인코딩 완료 및 파일 저장 성공: df_encoded_final.csv


### 2. 결측치(Unknown) 데이터의 명시적 라벨링
EDA 과정에서 확인한 'Unknown' 값들을 머신러닝 모델이 하나의 유의미한 정보로 인식할 수 있도록 텍스트 라벨링을 수행합니다.

비즈니스 목적: 정보 누락 자체가 가지는 잠재적 특성을 보존합니다.

분석 포인트: 단순 숫자로 되어 있는 Unknown(예: 7, 4, 6)을 문자열로 변환하여 원핫 인코딩 시 명확한 컬럼명을 가질 수 있도록 합니다.

In [ ]:
# [Why] 원본 데이터 보존을 위해 복사본을 생성하여 작업 (Deep Copy)
df_encoded = df.copy()

# [Why] 수치형으로 되어 있는 Unknown 값을 명시적인 문자열로 치환
# 이렇게 하면 get_dummies 수행 시 'education_Education_Unknown'과 같이 직관적인 컬럼명이 생성됩니다.
label_map = {
    'education': {7: 'Education_Unknown'},
    'marital': {4: 'Marital_Unknown'},
    'income': {6: 'Income_Unknown'}
}

for col, mapping in label_map.items():
    df_encoded[col] = df_encoded[col].replace(mapping)

print("✅ Unknown 상태 명시적 라벨링 완료")
display(df_encoded[['education', 'marital', 'income']].head())

### 3. 범주형 변수의 원핫 인코딩(One-Hot Encoding) 수행
모델이 이해할 수 있도록 범주형 변수를 수치형(0과 1)으로 변환합니다.

비즈니스 목적: 교육 수준, 결혼 여부 등 텍스트 데이터를 수학적 연산이 가능한 형태로 변환합니다.

기대 효과: 거리 기반 또는 트리 기반 알고리즘에서 범주형 변수의 영향을 정확히 반영할 수 있습니다.

In [ ]:
# [Why] 모델링에 사용할 범주형 컬럼 정의
# drop_first=False: 모든 범주를 컬럼으로 생성하여 특정 범주의 누락 없이 분석력을 높임 (필요 시 True로 변경 가능)
categorical_cols = ['education', 'marital', 'income', 'card_type']

# 원핫 인코딩 수행
df_final = pd.get_dummies(df_encoded, columns=categorical_cols, drop_first=False)

# [유지보수 팁] 새롭게 생성된 컬럼들의 데이터 타입이 bool인지 uint8인지 확인하세요.
# 최신 pandas 버전에서는 기본적으로 bool로 생성될 수 있으므로 필요 시 .astype(int) 변환이 필요할 수 있습니다.
print(f"✅ 원핫 인코딩 완료. 컬럼 수가 {len(df.columns)}개에서 {len(df_final.columns)}개로 확장되었습니다.")
display(df_final.head())

### 4. 최종 데이터셋 저장 및 검증
모델링 파이프라인의 다음 단계인 '학습' 단계로 넘어가기 전, 완성된 데이터를 물리적 파일로 저장합니다.

비즈니스 목적: 전처리 완료 데이터를 고정하여 실험의 재현성을 확보합니다.

유지보수 팁: 저장 시 index=False를 반드시 설정하여 불필요한 인덱스 컬럼이 중복 생성되는 것을 방지합니다.

In [ ]:
# [Why] 다음 단계(모델 학습 또는 클러스터링)에서 일관된 데이터를 사용하기 위해 CSV 저장
output_path = './data/df_encoded_final.csv'
df_final.to_csv(output_path, index=False)

# 최종 저장 결과 리포트
print(f"✅ 최종 데이터 저장 성공: {output_path}")
print(f"📊 최종 데이터셋 요약: {df_final.shape[0]} 행, {df_final.shape[1]} 열")